# 🚀 LLM Compressor: Scaling Study

This notebook analyzes how the number of **latent tokens (prompt length)** affects **accuracy** and **compression ratio** across different text lengths.

### Models to test:
- `HuggingFaceTB/SmolLM-135M` (Very fast, high quality for its size)
- `Qwen/Qwen2.5-0.5B` (Slightly larger, state-of-the-art for small models)
- `gpt2` (Baseline)

In [ ]:
# 1. Setup Environment
!pip install -q transformers accelerate torch
!git clone https://github.com/domguia/thinker || (cd thinker && git pull)
%cd thinker
import os
os.environ['PYTHONPATH'] = os.getcwd()

In [ ]:
# 2. Run the Scaling Study
# This will iterate through Short, Medium, and Long texts with varying prompt lengths.
!python scripts/study_scaling_curve.py --model "HuggingFaceTB/SmolLM-135M" --steps 150 --exp_name "smollm_scaling"

## 📊 Visualization
The results are logged in `dev_notes/compressor_experiments.md`. Let's visualize the curves.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

def parse_logs(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    data = []
    for line in lines:
        if '| 20' in line and 'Exp:' in line:
            parts = [p.strip() for p in line.split('|')]
            data.append({
                'target_len': int(parts[3]),
                'prompt_len': int(parts[4]),
                'accuracy': float(parts[6].replace('%', '')),
                'ratio': float(parts[7].replace('x', '')),
                'run': parts[8]
            })
    return pd.DataFrame(data)

df = parse_logs('dev_notes/compressor_experiments.md')

plt.figure(figsize=(12, 5))

# Plot Ratio vs Prompt Len
plt.subplot(1, 2, 1)
for tlen in df['target_len'].unique():
    subset = df[df['target_len'] == tlen].sort_values('prompt_len')
    plt.plot(subset['prompt_len'], subset['ratio'], marker='o', label=f'Target={tlen}')
plt.xlabel('Prompt Length (tokens)')
plt.ylabel('Compression Ratio')
plt.title('Compression Ratio vs Prompt Length')
plt.legend()
plt.grid(True)

# Plot Accuracy vs Prompt Len
plt.subplot(1, 2, 2)
for tlen in df['target_len'].unique():
    subset = df[df['target_len'] == tlen].sort_values('prompt_len')
    plt.plot(subset['prompt_len'], subset['accuracy'], marker='s', label=f'Target={tlen}')
plt.xlabel('Prompt Length (tokens)')
plt.ylabel('Accuracy (%)')
plt.title('Accuracy vs Prompt Length')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()